<p style="text-align:center"> 
    <a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/" target="_blank"> 
    <img src="../assets/logo.png" width="200" alt="Flavio Aguirre Logo"> 
    </a>
</p>

<h1 align="center"><strong>Weather Wise – 02 · Data Wrangling</strong></h1>
<hr>

In this notebook we perform **data wrangling** on the raw Australian weather dataset:

- Handle missing values (first, with a simple strategy).
- Adjust the problem definition to avoid data leakage.
- Focus on a **localized region** (Melbourne area) to capture consistent weather patterns.
- Engineer a simple **seasonality feature** from dates.
- Save a cleaned, ready–to–model dataset into `data/processed/`.

In [21]:
## Data Wrangling

import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

As we saw in the initial exploration, `Sunshine` and `Evaporation` look like important meteorological features,  
but they contain **many missing values**. For this first baseline, we will start with a **simplified approach** and later revisit more advanced strategies if needed.

In [22]:
# Load the dataset produced in 01_data-collection
df = pd.read_csv("../data/raw/weatherAUS-data.csv")

initial_shape = df.shape
print("Raw data loaded from ../data/raw/weatherAUS-data.csv")
print(f"Initial shape: {initial_shape[0]} rows × {initial_shape[1]} columns")

df.head()

Raw data loaded from ../data/raw/weatherAUS-data.csv
Initial shape: 145460 rows × 23 columns


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


### 1. Handling missing values (baseline approach)

For simplicity, we start by **dropping rows with missing values** and observing how much data remains.

> In a production setting, we would typically use **imputation** (e.g. median/mean for numerical features, most frequent category for categoricals) rather than dropping all rows with any missing value.  
> Here, the goal is to obtain a clean subset quickly and understand whether we still have enough data to train a baseline model.

In [23]:
# Number of rows before dropping missing values
rows_before = initial_shape[0]

# Drop rows with any missing values
df = df.dropna()

rows_after = df.shape[0]

print(f"Rows before dropna: {rows_before}")
print(f"Rows after dropna:  {rows_after}")
print(f"Rows removed:{rows_before - rows_after} ({(rows_before - rows_after) / rows_before:.1%} of original data)")

df.info()

Rows before dropna: 145460
Rows after dropna:  56420
Rows removed:89040 (61.2% of original data)
<class 'pandas.core.frame.DataFrame'>
Index: 56420 entries, 6049 to 142302
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           56420 non-null  object 
 1   Location       56420 non-null  object 
 2   MinTemp        56420 non-null  float64
 3   MaxTemp        56420 non-null  float64
 4   Rainfall       56420 non-null  float64
 5   Evaporation    56420 non-null  float64
 6   Sunshine       56420 non-null  float64
 7   WindGustDir    56420 non-null  object 
 8   WindGustSpeed  56420 non-null  float64
 9   WindDir9am     56420 non-null  object 
 10  WindDir3pm     56420 non-null  object 
 11  WindSpeed9am   56420 non-null  float64
 12  WindSpeed3pm   56420 non-null  float64
 13  Humidity9am    56420 non-null  float64
 14  Humidity3pm    56420 non-null  float64
 15  Pressure9am    56420 non-null  float64
 16

Since we still have **over 56,000 observations** after discarding rows with missing values,  
we can afford this aggressive approach for a **first baseline** without immediately resorting to complex imputation.

Later iterations of the project (see roadmap) can revisit this decision to recover more data.

In [24]:
# Quick sanity check of remaining columns
df.columns

Index(['Date', 'Location', 'MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation',
       'Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm',
       'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm',
       'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am',
       'Temp3pm', 'RainToday', 'RainTomorrow'],
      dtype='object')

## 2. Data leakage considerations

Before modeling, we must ensure that we **do not use information from the future** to predict the target.

Our original target is `RainTomorrow` (whether there is at least 1 mm of rain tomorrow).  
If we want to build a model that answers **"Should I take an umbrella today?"**, it is often more practical to:

- Predict **today’s rainfall** based on historical weather data **up to yesterday**.
- Avoid any accidental leakage of "tomorrow" information into our features.

To reflect this, we **rename the rainfall columns**:

- `RainToday` → `RainYesterday`  
- `RainTomorrow` → `RainToday` (new target label)

In [25]:
## Data Leak Considerations

# Ensure original columns exist before renaming
assert "RainToday" in df.columns, "'RainToday' column not found."
assert "RainTomorrow" in df.columns, "'RainTomorrow' column not found."

df = df.rename(
    columns={
        "RainToday": "RainYesterday",
        "RainTomorrow": "RainToday"
    }
)

print("Columns renamed to avoid leakage and clarify the prediction target:")
print("   - 'RainToday'     → 'RainYesterday'")
print("   - 'RainTomorrow'  → 'RainToday' (new target)")

Columns renamed to avoid leakage and clarify the prediction target:
   - 'RainToday'     → 'RainYesterday'
   - 'RainTomorrow'  → 'RainToday' (new target)


Now, the **modeling goal** becomes:

> Predict **`RainToday`** (at least 1 mm of rain today)  
> using weather conditions from **previous observations**, including `RainYesterday`.

## 3. Data granularity and location selection

Weather patterns are highly **location–dependent**.  
It is unrealistic to expect a single simple model to capture all the variability across **very different regions in Australia**.

We therefore focus on a **localized region** around Melbourne, where:

- Watsonia is only 15 km from Melbourne,
- Melbourne Airport is 18 km away.

By restricting the dataset to these locations, we aim for:

- More consistent underlying weather patterns,
- A simpler model that is still useful in practice (e.g. for local daily decisions).

In [26]:
## Location Selection

print("Unique locations in the full dataset:")
print(sorted(df["Location"].unique()))

# Filter to Melbourne area
melbourne_locations = ["Melbourne", "MelbourneAirport", "Watsonia"]
df = df[df["Location"].isin(melbourne_locations)]

print("\nFiltered to Melbourne area locations:", melbourne_locations)
df.info()

Unique locations in the full dataset:
['AliceSprings', 'Brisbane', 'Cairns', 'Canberra', 'Cobar', 'CoffsHarbour', 'Darwin', 'Hobart', 'Melbourne', 'MelbourneAirport', 'Mildura', 'Moree', 'MountGambier', 'NorfolkIsland', 'Nuriootpa', 'Perth', 'PerthAirport', 'Portland', 'Sale', 'Sydney', 'SydneyAirport', 'Townsville', 'WaggaWagga', 'Watsonia', 'Williamtown', 'Woomera']

Filtered to Melbourne area locations: ['Melbourne', 'MelbourneAirport', 'Watsonia']
<class 'pandas.core.frame.DataFrame'>
Index: 7557 entries, 64191 to 80997
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           7557 non-null   object 
 1   Location       7557 non-null   object 
 2   MinTemp        7557 non-null   float64
 3   MaxTemp        7557 non-null   float64
 4   Rainfall       7557 non-null   float64
 5   Evaporation    7557 non-null   float64
 6   Sunshine       7557 non-null   float64
 7   WindGustDir    7557 non-null   objec

We still have **7,557 records** after focusing on the Melbourne region,  
which is sufficient to train a reasonably good **localized** classification model.

More data (other cities or longer time spans) can always be incorporated in future iterations if needed.

## 4. Extracting a seasonality feature

Weather is inherently **seasonal**.  
We expect different rainfall patterns in **summer vs winter**, for example.

The dataset includes a `Date` column, which we can use to derive a **`Season`** feature.  
We will:

- Convert `Date` to a proper `datetime` type,
- Map each month to a season,
- Drop the raw `Date` column afterwards (to avoid collinearity and keep the feature set compact).

In [27]:
### Create a function to assign dates to seasons

def date_to_season(date):
    month = date.month
    if month in (12, 1, 2):
        return "Summer"
    elif month in (3, 4, 5):
        return "Autumn"
    elif month in (6, 7, 8):
        return "Winter"
    elif month in (9, 10, 11):
        return "Spring"

In [28]:
# Convert the "Date" column to datetime format
df["Date"] = pd.to_datetime(df["Date"])

# Apply the function to the "Date" column
df["Season"] = df["Date"].apply(date_to_season)

# Drop the original Date column (we keep only the derived seasonality)
df = df.drop(columns="Date")

df.head()

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainYesterday,RainToday,Season
64191,MelbourneAirport,11.2,19.9,0.0,5.6,8.8,SW,69.0,W,SW,...,37.0,1005.1,1006.4,7.0,7.0,15.9,18.1,No,Yes,Summer
64192,MelbourneAirport,7.8,17.8,1.2,7.2,12.9,SSE,56.0,SW,SSE,...,43.0,1018.0,1019.3,6.0,7.0,12.5,15.8,Yes,No,Summer
64193,MelbourneAirport,6.3,21.1,0.0,6.2,10.5,SSE,31.0,E,S,...,35.0,1020.8,1017.6,1.0,7.0,13.4,19.6,No,No,Summer
64194,MelbourneAirport,8.1,29.2,0.0,6.4,12.5,SSE,35.0,NE,SSE,...,23.0,1016.2,1012.8,5.0,4.0,16.0,28.2,No,No,Summer
64195,MelbourneAirport,9.7,29.0,0.0,7.4,12.3,SE,33.0,SW,SSE,...,31.0,1011.9,1010.3,6.0,2.0,19.4,27.1,No,No,Summer


In [29]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
MinTemp,7557.0,10.471589,4.480357,-2.1,7.3,10.1,13.6,30.5
MaxTemp,7557.0,20.698743,6.525832,8.4,15.6,19.4,24.6,46.8
Rainfall,7557.0,1.705836,4.993210,0.0,0.0,0.0,1.0,84.0
Evaporation,7557.0,4.666905,3.321487,0.0,2.2,4.0,6.4,23.8
Sunshine,7557.0,6.431878,3.894928,0.0,3.2,6.6,9.6,13.9
WindGustSpeed,7557.0,43.741829,15.606706,9.0,31.0,41.0,54.0,122.0
WindSpeed9am,7557.0,16.551145,10.821580,2.0,9.0,13.0,22.0,67.0
WindSpeed3pm,7557.0,20.133651,9.472907,2.0,13.0,19.0,26.0,76.0
Humidity9am,7557.0,71.933704,16.612418,11.0,62.0,72.0,84.0,100.0
Humidity3pm,7557.0,52.193992,17.635123,6.0,41.0,51.0,63.0,100.0


At this point, we have:

- A **cleaned, localized dataset** (Melbourne area only),
- No missing values (for this baseline),
- A new **`Season`** feature capturing coarse–grained seasonality,
- A clear target label: `RainToday` (after renaming).

This provides a solid starting point for the next steps:  
**EDA, feature encoding, and model training**.

In [30]:
### 5. Save the cleaned dataset

processed_dir = "../data/processed/"
os.makedirs(processed_dir, exist_ok=True)

output_path = os.path.join(processed_dir, "weatherAUS-data-clean.csv")
df.to_csv(output_path, index=False)

print(f"Data saved to {output_path}")

Data saved to ../data/processed/weatherAUS-data-clean.csv


<hr>

## Author

<a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/">**Flavio Aguirre**</a>
<br>
<a href="https://coursera.org/share/e27ae5af81b56f99a2aa85289b7cdd04">***Data Scientist***</a>